# Track B: BRCA1 Mutation Walk -- DNABERT-2 Embeddings

**Purpose**: Embed the BRCA1 single-point mutation walk using DNABERT-2 (pure Transformer
with BPE tokenization) and cache the results for cross-model analysis.

| Model | Architecture | Tokenization | Embedding |
|-------|-------------|-------------|-----------|
| **DNABERT-2** | BERT Transformer | BPE (multi-granularity) | Last hidden state mean-pool |

## Setup
1. Change Runtime to **GPU**
2. Run cells in order

Results are cached to `./results/interpolation_track_b/cache/` for the analysis notebook.

In [ ]:
# Cell: Install Dependencies
#
# DNABERT-2 needs triton for its custom FlashAttention.
# Also patches transformers 5.x compatibility issues.
# (Adapted from DNABERT2_RealDNA_Experiment.ipynb)

print('Installing core dependencies...')
!pip install -q torch transformers matplotlib seaborn pandas scipy biopython numpy scikit-learn
print('Installing triton (for DNABERT-2 FlashAttention)...')
!pip install -q triton

import transformers, torch
print(f'transformers {transformers.__version__}')
print(f'torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- Patch transformers 5.x for DNABERT-2 compatibility ---
from transformers import PretrainedConfig
import transformers.modeling_utils as _mu

_config_defaults = {
    'pad_token_id': None,
    'bos_token_id': None,
    'eos_token_id': None,
    'is_decoder': False,
    'add_cross_attention': False,
    'chunk_size_feed_forward': 0,
    'is_encoder_decoder': False,
    'tie_word_embeddings': True,
}
for attr, default in _config_defaults.items():
    if not hasattr(PretrainedConfig, attr):
        setattr(PretrainedConfig, attr, default)
        print(f'Patched config default: {attr}={default}')

# Patch all_tied_weights_keys for models that don't define it
if not getattr(_mu, '_dnabert2_patched', False):
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._dnabert2_patched = True
    print('Patched: all_tied_weights_keys / tie_weights')

# Disable meta-device initialization
# transformers 5.x + accelerate uses init_empty_weights() to create models
# on a 'meta' device first (lazy tensors). DNABERT-2's custom code does real
# tensor ops during __init__ which crashes on meta tensors.
from contextlib import contextmanager

@contextmanager
def _no_meta_init(**kwargs):
    yield

try:
    import accelerate
    accelerate.init_empty_weights = _no_meta_init
    print('Patched: accelerate.init_empty_weights (disabled meta init)')
except ImportError:
    pass

import sys
for mod_name in ['transformers.modeling_utils', 'transformers.integrations.accelerate']:
    mod = sys.modules.get(mod_name)
    if mod and hasattr(mod, 'init_empty_weights'):
        mod.init_empty_weights = _no_meta_init
        print(f'Patched: {mod_name}.init_empty_weights')

print('\nDone!')

In [ ]:
# Cell: Configuration

import os
import sys
import numpy as np
import torch

sys.path.insert(0, '.')

# --- Output paths ---
OUTPUT_BASE = './results/interpolation_track_b/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# --- BRCA1 ---
BRCA1_REGION_LEN = 2000   # 2kb core region for mutation walk
BRCA1_FLANKING   = 16_384 # Total region to fetch (with flanking context)
N_EXTRA_SNPS     = 120
SEED             = 320

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_BASE}')

# --- DNABERT-2 ---
DNABERT2_MODEL = 'zhihan1996/DNABERT-2-117M'
print(f'DNABERT-2 model: {DNABERT2_MODEL}')

In [ ]:
# Cell: Download BRCA1 Region & Define Mutation Walk Endpoints

import urllib.request
import json as _json


def fetch_brca1_region(total_len=BRCA1_FLANKING, core_len=BRCA1_REGION_LEN):
    '''Fetch BRCA1 region from UCSC (GRCh38/hg38).

    Downloads total_len bp centered on the C61G variant position.
    The central core_len bp is the "mutation zone" where SNPs are introduced.
    '''
    center = 43_104_121  # BRCA1 C61G position (GRCh38)
    start = center - total_len // 2
    end = start + total_len
    core_start = (total_len - core_len) // 2
    core_end = core_start + core_len

    url = (f'https://api.genome.ucsc.edu/getData/sequence'
           f'?genome=hg38;chrom=chr17;start={start};end={end}')

    print(f'Fetching BRCA1 region chr17:{start:,}-{end:,} ({total_len:,} bp)...')
    try:
        req = urllib.request.Request(url)
        req.add_header('User-Agent', 'GeometricTax-Experiment/1.0')
        with urllib.request.urlopen(req, timeout=30) as response:
            data = _json.loads(response.read().decode())
            seq = data.get('dna', '').upper()
            if len(seq) >= total_len * 0.9:
                seq = seq[:total_len]
                rng = np.random.default_rng(SEED)
                seq = ''.join(
                    c if c in 'ACGT' else rng.choice(list('ACGT'))
                    for c in seq
                )
                print(f'  Downloaded {len(seq)} bp from UCSC')
                print(f'  Core region: positions [{core_start}:{core_end}] ({core_len} bp)')
                return seq, core_start, core_end
    except Exception as e:
        print(f'  UCSC download failed: {e}')

    # Fallback: synthetic
    print('  Generating synthetic BRCA1-like sequence as fallback...')
    rng = np.random.default_rng(SEED + 17)
    bases = rng.choice(list('ACGT'), size=total_len, p=[0.29, 0.21, 0.21, 0.29])
    seq = ''.join(bases)
    print(f'  Generated {len(seq)} bp synthetic BRCA1-like sequence')
    return seq, core_start, core_end


def create_pathogenic_variant(full_seq, core_start, core_end,
                              n_extra_snps=N_EXTRA_SNPS, seed=SEED):
    '''Create a pathogenic variant by mutating only the core region.'''
    rng = np.random.default_rng(seed)
    bases = list(full_seq)
    core_center = (core_start + core_end) // 2
    mutated_positions = set()

    # 1. Pathogenic mutation at core center
    alt_base = 'G' if bases[core_center] != 'G' else 'C'
    print(f'  Pathogenic mutation at position {core_center}: {bases[core_center]} -> {alt_base}')
    bases[core_center] = alt_base
    mutated_positions.add(core_center)

    # 2. Additional random SNPs within core region only
    available = [i for i in range(core_start, core_end) if i not in mutated_positions]
    snp_positions = rng.choice(available, size=min(n_extra_snps, len(available)),
                                replace=False)
    snp_positions.sort()

    for pos in snp_positions:
        original = bases[pos]
        alternatives = [b for b in 'ACGT' if b != original]
        bases[pos] = rng.choice(alternatives)
        mutated_positions.add(pos)

    mutant = ''.join(bases)
    hamming = sum(a != b for a, b in zip(full_seq, mutant))
    print(f'  Total mutations: {hamming} (all within core [{core_start}:{core_end}])')
    return mutant, sorted(mutated_positions)


# Generate endpoints
full_wt_seq, CORE_START, CORE_END = fetch_brca1_region()
full_mut_seq, mutation_positions = create_pathogenic_variant(
    full_wt_seq, CORE_START, CORE_END)

print(f'\nFull sequence length: {len(full_wt_seq)}')
print(f'Core region: [{CORE_START}:{CORE_END}] ({CORE_END - CORE_START} bp)')
print(f'Hamming distance: {sum(a != b for a, b in zip(full_wt_seq, full_mut_seq))}')

wildtype_seq = full_wt_seq
mutant_seq = full_mut_seq

In [ ]:
# Cell: Generate Single-Point Mutation Walk

def single_point_mutation_walk(seq_start, seq_end, seed=SEED):
    """Create a mutation walk from seq_start to seq_end, one base at a time."""
    assert len(seq_start) == len(seq_end), 'Sequences must be same length'

    diff_positions = [i for i in range(len(seq_start))
                      if seq_start[i] != seq_end[i]]

    rng = np.random.default_rng(seed)
    rng.shuffle(diff_positions)

    current = list(seq_start)
    walk = [''.join(current)]
    step_positions = []

    for pos in diff_positions:
        current[pos] = seq_end[pos]
        walk.append(''.join(current))
        step_positions.append(pos)

    assert walk[-1] == seq_end, 'Walk did not reach target'
    return walk, step_positions


mutation_walk, walk_positions = single_point_mutation_walk(wildtype_seq, mutant_seq)

n_steps = len(mutation_walk)
print(f'Mutation walk: {n_steps} steps (start + {n_steps - 1} mutations)')

center_pos = (CORE_START + CORE_END) // 2
pathogenic_step = None
for i, pos in enumerate(walk_positions):
    if pos == center_pos:
        pathogenic_step = i + 1
        break
print(f'Pathogenic C61G mutation occurs at step {pathogenic_step}/{n_steps - 1}')

walk_cache = f'{CACHE_DIR}/brca1_mutation_walk.npz'
np.savez(walk_cache, walk=mutation_walk, positions=walk_positions,
         pathogenic_step=pathogenic_step)
print(f'Walk cached to {walk_cache}')

In [ ]:
# Cell: Load DNABERT-2
#
# CRITICAL: Cannot use AutoModel.from_pretrained with DNABERT-2 on
# transformers 5.x + accelerate. The meta-tensor initialization path
# crashes because DNABERT-2's custom bert_layers.py does real tensor
# ops during __init__.
#
# Manually resolve the model class via HF's dynamic module utils,
# instantiate on CPU, then load state dict directly.
# Also patches flash_attn_triton.py for Triton >= 3.x compatibility.
# (Adapted from DNABERT2_RealDNA_Experiment.ipynb)

import torch
import gc
import sys
import os
import re
from transformers import AutoTokenizer, AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module
from huggingface_hub import hf_hub_download


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_dnabert2(model_name=DNABERT2_MODEL, batch_size=8):
    """Load DNABERT-2 via manual class resolution + state dict loading."""
    print(f'Loading {model_name}...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # Step 1: Load config
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # Step 2: Patch flash_attn_triton.py for Triton >= 3.x
    flash_path = hf_hub_download(model_name, 'flash_attn_triton.py')
    with open(flash_path, 'r') as f:
        source = f.read()
    if 'trans_b=True' in source:
        source = re.sub(
            r'tl\.dot\((\w+),\s*(\w+),\s*trans_b\s*=\s*True\)',
            r'tl.dot(\1, tl.trans(\2))',
            source
        )
        with open(flash_path, 'w') as f:
            f.write(source)
        for key in list(sys.modules.keys()):
            if 'flash_attn_triton' in key or 'bert_layers' in key:
                del sys.modules[key]
        print('  Patched flash_attn_triton.py for Triton >= 3.x')
    else:
        print('  flash_attn_triton.py already compatible')

    # Step 3: Resolve the custom BertModel class
    auto_map = getattr(config, 'auto_map', {})
    class_ref = auto_map.get('AutoModel', auto_map.get('AutoModelForMaskedLM', 'bert_layers.BertModel'))
    print(f'  Resolving class: {class_ref}')
    BertModel = get_class_from_dynamic_module(class_ref, model_name)
    print(f'  Resolved: {BertModel.__module__}.{BertModel.__name__}')

    # Step 4: Create model on CPU (no meta tensors)
    print('  Initializing model on CPU...')
    model = BertModel(config)

    # Step 5: Load pretrained weights
    weight_file = hf_hub_download(model_name, 'pytorch_model.bin')
    print('  Loading pretrained weights...')
    state_dict = torch.load(weight_file, map_location='cpu', weights_only=False)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'  Note: {len(missing)} missing keys (normal for base model)')
    if unexpected:
        print(f'  Note: {len(unexpected)} unexpected keys')

    model = model.to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB')
    else:
        print(f'  Params: {n_params:.1f}M')

    max_length = tokenizer.model_max_length
    if max_length > 100000:
        max_length = 512
    print(f'  Max token length: {max_length}')

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 10 == 0 or batch_idx == n_batches:
                print(f'  Batch {batch_idx}/{n_batches}', end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens)
            hidden = outputs[0]  # DNABERT-2 returns tuple, not NamedTuple

            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())
        print()
        return np.concatenate(embeddings, axis=0).astype(np.float32)

    print(f'DNABERT-2 ready: {n_params:.1f}M params')
    return embed, model, tokenizer, n_params


dnabert2_embed, dnabert2_model, dnabert2_tokenizer, dnabert2_params = load_dnabert2()

In [ ]:
# Cell: Embed Mutation Walk with DNABERT-2
#
# DNABERT-2 has a 512-token max length. With BPE tokenization on DNA,
# this covers roughly 1-2 kb depending on the learned merges.
# We truncate sequences to the core region to stay within limits.

dnabert2_cache = f'{CACHE_DIR}/dnabert2_brca1_walk.npy'

if os.path.exists(dnabert2_cache):
    print('Loading cached DNABERT-2 embeddings...')
    dnabert2_embeddings = np.load(dnabert2_cache)
else:
    # DNABERT-2 has limited context -- use core region centered on mutations
    # to ensure the walk mutations are visible to the model
    core_walk = [seq[CORE_START:CORE_END] for seq in mutation_walk]
    print(f'Embedding {len(core_walk)} walk steps with DNABERT-2...')
    print(f'(Using core region [{CORE_START}:{CORE_END}], {CORE_END - CORE_START} bp)')

    # Check how many tokens the core region produces
    test_tokens = dnabert2_tokenizer(core_walk[0], return_tensors='pt')
    n_tok = test_tokens['input_ids'].shape[1]
    print(f'Core region tokenizes to ~{n_tok} BPE tokens')

    dnabert2_embeddings = dnabert2_embed(core_walk)
    np.save(dnabert2_cache, dnabert2_embeddings)
    print(f'Cached to {dnabert2_cache}')

print(f'DNABERT-2 embeddings: {dnabert2_embeddings.shape}')

# Free GPU memory
del dnabert2_model
cleanup_gpu()
print('DNABERT-2 model freed from GPU')

In [ ]:
# Cell: Quick Sanity Check

from scipy.spatial.distance import cosine

print('Sanity check: cosine distance from wildtype')
for step_idx in [0, 10, 50, len(dnabert2_embeddings) - 1]:
    if step_idx < len(dnabert2_embeddings):
        d = cosine(dnabert2_embeddings[0], dnabert2_embeddings[step_idx])
        print(f'  Step {step_idx:4d}: cosine dist = {d:.6f}')

# Consecutive step distances
dists = [cosine(dnabert2_embeddings[i], dnabert2_embeddings[i + 1])
         for i in range(len(dnabert2_embeddings) - 1)]
dists = np.array(dists)
print(f'\nConsecutive step cosine distances:')
print(f'  mean={dists.mean():.6f}, std={dists.std():.6f}, max={dists.max():.6f}')
print(f'  smoothness (mean/max): {dists.mean() / (dists.max() + 1e-8):.4f}')
print(f'\nDNABERT-2 embeddings cached and ready for analysis notebook.')